# 🚀 Training Dual-Stream YOLOv11 di Kaggle dengan Dataset Primer (hujan-2)

Notebook ini digunakan untuk melatih arsitektur **Dual-Stream YOLOv11 (DE-YOLOv11)** menggunakan GPU (Tesla T4 x2 / P100) di Kaggle.

### 1. Cek GPU & Install Dependensi

In [ ]:
# Cek ketersediaan GPU
!nvidia-smi

# Install pustaka Ultralytics
!pip install -q ultralytics

### 2. Clone Repositori Dual-Stream YOLOv11 dari GitHub

In [ ]:
import os

%cd /kaggle/working

# Clone repo jika belum ada
if not os.path.exists('Dual-Stream-YOLOv11'):
    !git clone https://github.com/adityanhh/Dual-Stream-YOLOv11.git

%cd /kaggle/working/Dual-Stream-YOLOv11
print("✅ Berhasil masuk ke repositori Dual-Stream-YOLOv11!")

### 3. Eksplorasi Struktur Folder Dataset `hujan-2` di Kaggle

In [ ]:
from pathlib import Path

dataset_root = Path("/kaggle/input/hujan-2")
print(f"Isi folder {dataset_root}:")

if dataset_root.exists():
    for item in sorted(dataset_root.iterdir()):
        if item.is_dir():
            print(f" 📁 {item.name}/ ({len(list(item.iterdir()))} sub-items)")
            for sub in sorted(item.iterdir())[:5]:
                print(f"    └── {sub.name}")
        else:
            print(f" 📄 {item.name}")
else:
    print("⚠️ Dataset hujan-2 belum ditambahkan di Input! Klik 'Add Input' di kanan atas dan pilih dataset 'hujan-2'.")

### 4. Buat File Konfigurasi `hujan2_kaggle.yaml`

In [ ]:
import yaml

# Konfigurasi dataset hujan-2 di Kaggle
yaml_data = {
    'path': '/kaggle/input/hujan-2',
    'train_vis': 'images/vis_train',
    'train_ir': 'images/Ir_train',
    'train_labels': 'labels/vis_train',
    'val_vis': 'images/vis_val',
    'val_ir': 'images/Ir_val',
    'val_labels': 'labels/vis_val',
    'nc': 1,
    'names': {
        0: 'person'
    }
}

yaml_save_path = '/kaggle/working/hujan2_kaggle.yaml'
with open(yaml_save_path, 'w') as f:
    yaml.dump(yaml_data, f)

print(f"✅ File konfigurasi dataset berhasil disimpan di: {yaml_save_path}")

### 5. Download Pretrained Weights YOLO11 (Transfer Learning)

In [ ]:
# Download bobot pretrained yolo11n.pt untuk mempercepat proses konvergensi
!wget -q https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt
print("✅ Bobot yolo11n.pt berhasil di-download!")

### 6. Jalankan Pelatihan (Training) Dual-Stream YOLOv11

In [ ]:
!python train.py \
    --model configs/yolo11n-dualstream.yaml \
    --data /kaggle/working/hujan2_kaggle.yaml \
    --pretrained yolo11n.pt \
    --epochs 100 \
    --batch 16 \
    --imgsz 640 \
    --device 0 \
    --project /kaggle/working/runs/train \
    --name exp_hujan2

### 7. Uji Inferensi (Prediction) pada Pasangan Citra Uji

In [ ]:
import glob

# Cari contoh citra validasi
vis_samples = sorted(glob.glob('/kaggle/input/hujan-2/images/vis_val/*.jpg')) + sorted(glob.glob('/kaggle/input/hujan-2/images/vis_val/*.png'))

if vis_samples:
    sample_vis = vis_samples[0]
    sample_ir = sample_vis.replace('vis_val', 'Ir_val')
    
    !python predict.py \
        --weights /kaggle/working/runs/train/exp_hujan2/best.pt \
        --model configs/yolo11n-dualstream.yaml \
        --vis-img "{sample_vis}" \
        --ir-img "{sample_ir}" \
        --conf 0.25 \
        --save-dir /kaggle/working/runs/predict
else:
    print("ℹ️ Tidak menemukan file di vis_val, silakan masukkan path gambar uji secara manual.")

### 8. Kompres & Download Hasil Pelatihan (`best.pt` & Checkpoints)

In [ ]:
import shutil

# Kompres seluruh folder hasil pelatihan menjadi zip
shutil.make_archive('/kaggle/working/hujan2_training_results', 'zip', '/kaggle/working/runs/train/exp_hujan2')
print("🎉 Selesai! File 'hujan2_training_results.zip' sudah dibuat di /kaggle/working dan siap di-download!")